# 🧠 Agentic RAG Pipeline — End-to-End Demo

**Assignment 3 — RAG Architect 2026**

This notebook demonstrates the full Agentic RAG pipeline combining:
- **Adaptive RAG** — 3-way query routing (vectorstore / direct_llm / web_search)
- **Corrective RAG** — document grading, query rewriting, web search fallback
- **Self-RAG** — hallucination checking + answer quality grading with retry loop

**Stack:** LangGraph · AWS Bedrock (Claude 3 Haiku + Titan Embeddings) · FAISS · Tavily · SQLite caching

## 0. Setup

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv()

from app.config import VECTOR_STORE_PATH, CATEGORY_KEYWORDS
from app.embedding import load_vector_store
from src.retrieval import retrieve_documents, rerank_documents
from src.generation import generate_answer
from src.pipeline import RAGPipeline
from src.agentic.graph import build_agentic_rag_graph, run_agentic_rag

# Initialize pipeline and load vector store
pipeline = RAGPipeline()
vectorstore = load_vector_store()

if vectorstore:
    print(f'✅ Vector store loaded from {VECTOR_STORE_PATH}')
    print(f'📂 Knowledge base categories: {list(CATEGORY_KEYWORDS.keys())}')
else:
    print('❌ No vector store found — run Streamlit and click Rebuild Knowledge Base first')

In [ ]:
# Verify graph compiles
graph = build_agentic_rag_graph()
print('✅ LangGraph agentic RAG graph compiled successfully')
print(f'   Nodes: {list(graph.nodes.keys())}')

In [ ]:
# Helper to run a query and display results
def demo_query(question, chat_history=None, label=''):
    """Run agentic pipeline and display trace + answer."""
    print(f'\n{"="*70}')
    if label:
        print(f'🏷️  {label}')
    print(f'❓ Question: {question}')
    print(f'{"="*70}\n')
    
    result = pipeline.agentic_query(
        question=question,
        vectorstore=vectorstore,
        retriever_func=retrieve_documents,
        reranker_func=rerank_documents,
        generator_func=generate_answer,
        chat_history=chat_history,
    )
    
    # Display trace
    node_icons = {
        'cache': '💾', 'router': '🔀', 'rewriter': '✏️', 'retrieve': '📥',
        'grader': '✅', 'rerank': '📊', 'generate': '⚡', 'direct_llm': '💬',
        'fallback': '⚠️', 'hallucination_check': '🔍', 'answer_grader': '🎯',
        'corrective_rewrite': '🔄', 'web_search': '🌐',
    }
    
    print('📍 Agent Trace:')
    for step in result.get('trace', []):
        icon = node_icons.get(step['node'], '⚙️')
        print(f"   {icon} {step['node']} → {step['decision']}")
        print(f"      {step.get('reason', '')}")
    
    print(f'\n🛤️  Route: {result.get("route", "unknown")}')
    print(f'🔁 Retries: {result.get("retry_count", 0)}')
    print(f'💾 From cache: {result.get("from_cache", False)}')
    print(f'\n💡 Answer:\n{result["answer"]}')
    
    return result

---
## Branch 1: Vectorstore Retrieval (Adaptive → Corrective → Self-RAG)

Questions about internal knowledge base topics are routed to the **vectorstore** path.
The pipeline: Router → Rewriter → Retrieve → Grade Docs → Rerank → Generate → Hallucination Check → Answer Grade

In [ ]:
# 1a. Single-category vectorstore query
r1 = demo_query(
    'What is the Avaya project about?',
    label='BRANCH 1a: Vectorstore — Single Category'
)

In [ ]:
# 1b. Multi-category comparison query (tests balanced retrieval)
r2 = demo_query(
    'What is the Avaya project and how is it different from the DOT project?',
    label='BRANCH 1b: Vectorstore — Multi-Category Comparison'
)

In [ ]:
# 1c. Follow-up question (tests conversational rewriter)
chat_history = [
    {'role': 'user', 'content': 'What tables are in the BPPSL project?'},
    {'role': 'assistant', 'content': 'The BPPSL tables include: BKNG_PNR_PAX_SEG_LEG, ITIN_FLT_PATH, TEMP_LOAD_DEFAULT_FARE, DEFAULT_FARE.'},
]
r3 = demo_query(
    'Explain the first one in detail',
    chat_history=chat_history,
    label='BRANCH 1c: Vectorstore — Follow-up with Chat History'
)

---
## Branch 2: Direct LLM (No Retrieval)

Greetings, chitchat, and general knowledge questions are routed to the **direct_llm** path.
The pipeline: Router → Direct Generate → END (no retrieval, no grading)

In [ ]:
# 2a. Greeting
r4 = demo_query(
    'Hello! What can you help me with?',
    label='BRANCH 2a: Direct LLM — Greeting'
)

In [ ]:
# 2b. General knowledge (not in KB)
r5 = demo_query(
    'What is the capital of France?',
    label='BRANCH 2b: Direct LLM — General Knowledge'
)

---
## Branch 3: Web Search Fallback (Tavily)

Questions requiring current/external information are routed to **web_search**.
The pipeline: Router → Web Search (Tavily) → Generate → Hallucination Check → Answer Grade

> ⚠️ Requires `TAVILY_API_KEY` in `.env`. If not set, the node will skip gracefully.

In [ ]:
# 3a. Current events / external knowledge
r6 = demo_query(
    'What are the latest features in LangGraph 2025?',
    label='BRANCH 3a: Web Search — External Knowledge'
)

In [ ]:
# 3b. Another web search query
r7 = demo_query(
    'What is the current price of AWS Bedrock Claude 3 Haiku?',
    label='BRANCH 3b: Web Search — API Pricing'
)

---
## Corrective RAG Demo: Query Rewrite + Fallback

When all retrieved docs are graded irrelevant, the pipeline:
1. Rewrites the query (corrective rewriter)
2. Retries retrieval
3. If still irrelevant → falls back to web search or parametric LLM

In [ ]:
# This query is intentionally vague to trigger corrective rewriting
r8 = demo_query(
    'Tell me about the migration process',
    label='CORRECTIVE RAG: Vague Query → Rewrite → Retry'
)

---
## Caching Demo

The 3-tier cache system (exact → semantic → retrieval) avoids redundant LLM calls.

In [ ]:
# First call — cache miss, full pipeline
print('--- First call (cache miss) ---')
r9a = demo_query(
    'What is BPPSL?',
    label='CACHE: First call — full pipeline'
)

# Second call — should hit exact or semantic cache
print('\n--- Second call (cache hit expected) ---')
r9b = demo_query(
    'What is BPPSL?',
    label='CACHE: Repeat call — cache hit expected'
)

In [ ]:
# Cache statistics
stats = pipeline.get_cache_stats()
print('📊 Cache Statistics:')
for tier, s in stats.items():
    if tier != 'overall':
        print(f"   {tier}: {s['hits']} hits / {s['misses']} misses (rate: {s['hit_rate']:.1%}, size: {s['size']})")
if 'overall' in stats:
    print(f"   Overall: {stats['overall']['hit_rate']:.1%} hit rate across {stats['overall']['total_requests']} requests")

---
## Summary

| Branch | Route | Nodes Traversed | RAG Layers |
|--------|-------|-----------------|------------|
| KB Query | vectorstore | Router → Rewriter → Retrieve → Grade → Rerank → Generate → Hallucination → Answer Grade | Adaptive + Corrective + Self-RAG |
| Greeting | direct_llm | Router → Direct Generate | Adaptive |
| External | web_search | Router → Web Search → Generate → Hallucination → Answer Grade | Adaptive + Self-RAG |
| Failed Retrieval | vectorstore → fallback | Router → Retrieve → Grade → Corrective Rewrite → Retrieve → Web Search | Adaptive + Corrective |
| Cached | cache hit | Cache → (skip pipeline) | Cache tier |